To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

<a name="top"></a>

## Table of Contents

1. [Foundations — Embeddings, Fine-Tuning, Unsloth & GPU Setup](#foundations)
2. [Architecture — EmbeddingGemma 300M Under the Hood](#architecture)
3. [Installation](#installation)
4. [Model Loading](#model-loading)
5. [LoRA — Theory, Math & Parameter Reference](#lora)
6. [Data Preparation](#data)
7. [Baseline Evaluation](#baseline)
8. [Loss Functions — The Complete Catalog](#loss-functions)
9. [Training Configuration](#train)
10. [Training Internals — What Happens Under the Hood](#training-internals)
11. [Post-Training Evaluation](#post-eval)
12. [Evaluation Metrics Explained](#metrics-explained)
13. [Inference](#inference)
14. [Saving & Deployment](#saving-overview)
    - [Option 1: LoRA Adapters Only](#save-lora)
    - [Option 2: Merged Float16](#save-merged)
    - [Option 3: GGUF / llama.cpp](#save-gguf)
15. [Production Considerations](#production)
16. [Troubleshooting Common Issues](#troubleshooting)
17. [Glossary](#glossary)

### News

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

You can now train embedding models 1.8-3.3x faster with 20% less VRAM. [Blog](https://unsloth.ai/docs/new/embedding-finetuning)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

3x faster LLM training with 30% less VRAM and 500K context. [3x faster](https://unsloth.ai/docs/new/3x-faster-training-packing) • [500K Context](https://unsloth.ai/docs/new/500k-context-length-fine-tuning)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

<a name="foundations"></a>

## 1. Foundations — Embeddings, Fine-Tuning, Unsloth & GPU Setup

### What Is an Embedding Model?

An embedding model converts text into a fixed-length vector of numbers (an "embedding"). For example, the sentence *"The patient has chest pain"* might become a 768-dimensional vector like `[0.23, -0.87, 0.45, ...]`.

The critical property: **semantically similar texts produce similar vectors.** "The patient has chest pain" and "Chest discomfort reported" will have vectors that are close together in 768-dimensional space, while "The weather is sunny" will be far away.

**Common use cases:**
- **Semantic search** — find documents relevant to a query without keyword matching.
- **Retrieval-Augmented Generation (RAG)** — feed relevant context to an LLM.
- **Clustering** — group similar documents automatically.
- **Classification** — use embeddings as features for downstream classifiers.
- **Duplicate detection** — find near-duplicate content.

### What Is Fine-Tuning?

A pre-trained model like EmbeddingGemma 300M has been trained on massive general-purpose data. It understands language broadly but not your specific domain. Fine-tuning takes this pre-trained model and continues training it on **your** data so it learns domain-specific patterns.

Think of it like this: the pre-trained model is a medical student who has read every textbook. Fine-tuning is their residency, where they learn the specific patterns of your hospital's patient population.

### What Is Unsloth?

[Unsloth](https://unsloth.ai) is an open-source library that makes fine-tuning faster and more memory-efficient. It achieves this through five key optimizations:

1. **Custom CUDA kernels** for LoRA parameter updates that fuse matrix multiplications.
2. **Smart gradient checkpointing** that offloads activations to CPU pinned memory.
3. **Automatic `torch.compile` integration** for encoder models (up to 6x speedup).
4. **Optimized attention** via Flash Attention 2 or SDPA (Scaled Dot-Product Attention).
5. **Memory-efficient saving** with intelligent RAM management during model export.

Without Unsloth, fine-tuning EmbeddingGemma 300M would use more GPU memory and take roughly 2-3x longer.

### Choosing Your GPU

Any NVIDIA GPU with CUDA support can run this notebook. Here's what to expect:

| GPU | VRAM | bfloat16 | Batch Size Guidance |
|-----|------|----------|---------------------|
| **T4** (Colab free) | 15 GB | No (fp16) | 32–64 |
| **L4** (Colab paid) | 24 GB | Yes | 64–128 |
| **A100** (cloud) | 40/80 GB | Yes | 128–256 |
| **RTX 3060** | 12 GB | Yes | 16–32 |
| **RTX 3090** | 24 GB | Yes | 64–128 |
| **RTX 4070 Ti / 4080** | 16 GB | Yes | 32–64 |
| **RTX 4090** | 24 GB | Yes | 64–128 |
| **RTX 5090** | 32 GB | Yes | 128–256 |

**Key thresholds:**
- **Compute capability >= 8.0** (Ampere+): supports bfloat16, which is more numerically stable for training.
- **Compute capability 7.x** (Turing — T4, RTX 2000 series): bfloat16 not supported; the notebook falls back to float16, which works fine.
- **12 GB VRAM minimum**: The model uses ~3.5 GB; you need headroom for activations, gradients, and optimizer states. Reduce batch size on 8 GB cards.

**Google Colab** is the easiest way to get started — the free tier gives you a T4 with 15 GB VRAM, enough for the full notebook with `batch_size=64`.

### Installing PyTorch for Your CUDA Version

**On Google Colab:** PyTorch comes pre-installed with the correct CUDA version. Skip this.

**On a local machine:** You must install a PyTorch build that matches your system's CUDA toolkit. This is the #1 cause of "CUDA not found" errors.

```bash
# Step 1: Check your CUDA version
nvidia-smi   # Look for "CUDA Version" in the top-right corner

# Step 2: Install matching PyTorch (pick ONE)
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118  # CUDA 11.8
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121  # CUDA 12.1
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124  # CUDA 12.4
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126  # CUDA 12.6

# Step 3: Verify
python -c "import torch; print(torch.cuda.is_available())"  # Must be True
```

If `torch.cuda.is_available()` returns `False`, you installed the CPU-only build (plain `pip install torch` defaults to CPU). Reinstall with the `--index-url` flag above.

##Import Data from Drive


In [24]:
from google.colab import drive
import pandas as pd
#

In [25]:
file_path ="/content/drive/MyDrive/MCRC '25-'26/DoD SAFE-gnRaRjhqoBbY9jcy/GenAI Folder/MCRC Volumes/CSV DATA/query_passage_dataset_Operations.pdf_20260226_222157.csv"


In [26]:
df = pd.read_csv(file_path)

# 2. Filter for label = 1 and select/rename columns
# This keeps only the 'positive' matches and renames them to your format
df_filtered = df[df['label'] == 1][['query', 'passage']].rename(columns={
    'query': 'question',
    'passage': 'passage_text'
})

# 3. Convert the filtered data to a Hugging Face Dataset
# preserve_index=False prevents pandas indices from becoming a separate column
full_dataset = Dataset.from_pandas(df_filtered, preserve_index=False)
# 4. Create train and eval splits
# You can either take specific counts (like 10,000 and 2,000) or use a percentage
total_available = len(full_dataset)

split_data = full_dataset.train_test_split(test_size=0.2)

train_dataset = split_data['train']
eval_dataset = split_data['test']

# Verification
print(f"Total available rows (label=1): {len(full_dataset)}")
print(f"Train size: {len(train_dataset)}")  # Should be ~7,270
print(f"Eval size: {len(eval_dataset)}")    # Should be ~1,818

Total available rows (label=1): 9088
Train size: 7270
Eval size: 1818


In [27]:
# Verification
print(f"Total positive rows: {total_available}")
print(f"Train size: {len(train_dataset)}")
print(f"Eval size: {len(eval_dataset)}")
print(f"Sample: {train_dataset[0]}")

Total positive rows: 9088
Train size: 7270
Eval size: 1818
Sample: {'question': '**What is the primary condition necessary for a company to meet its monthly quotas?', 'passage_text': 'Monthly quotas cannot be met without proper placement of contracts in the required months.'}


<a name="architecture"></a>

## 2. Architecture — EmbeddingGemma 300M Under the Hood

Before we load the model, let's understand what we're working with.

### Model Specifications

| Property | Value |
|----------|-------|
| **Architecture** | Gemma 3 (bidirectional encoder) |
| **Parameters** | ~316 million |
| **Hidden Dimension** | 768 |
| **Transformer Layers** | 24 |
| **Attention Heads** | 3 (Grouped Query Attention with 1 KV head) |
| **Head Dimension** | 256 |
| **MLP Intermediate Size** | 1,152 |
| **Vocabulary Size** | 262,144 tokens |
| **Max Position Embeddings** | 2,048 tokens |
| **Attention Type** | Bidirectional (not causal) |
| **Position Encoding** | Rotary Position Embeddings (RoPE) |
| **Output Embedding Dimension** | 768 |

### Bidirectional vs. Causal Attention

A generative model like Gemma 3 uses **causal (unidirectional) attention**: each token can only attend to tokens before it, enabling next-token prediction.

EmbeddingGemma uses **bidirectional attention**: each token attends to **all** other tokens, both before and after it. This produces richer representations because every token has full context of the entire input. The critical config flag is `"use_bidirectional_attention": true`. Because of this, EmbeddingGemma cannot generate text — it only produces embeddings.

### The 24-Layer Transformer Structure

Each of the 24 layers contains:

1. **RMSNorm** (Root Mean Square Layer Normalization)
2. **Self-Attention** with:
   - `q_proj`: Query projection (768 → 768, split into 3 heads of 256 each)
   - `k_proj`: Key projection (768 → 256, 1 KV head)
   - `v_proj`: Value projection (768 → 256, 1 KV head)
   - Per-head RMSNorm on queries and keys
   - RoPE (Rotary Position Embeddings)
   - `o_proj`: Output projection (768 → 768)
3. **RMSNorm**
4. **MLP (Feed-Forward Network)** using SwiGLU:
   - `gate_proj`: 768 → 1,152 (gating mechanism)
   - `up_proj`: 768 → 1,152 (value pathway)
   - Element-wise multiply: `GELU(gate_proj(x)) * up_proj(x)`
   - `down_proj`: 1,152 → 768 (project back)

### Hybrid Sliding Window + Full Attention

EmbeddingGemma uses a hybrid attention pattern:
- **20 layers** use sliding window attention (window size = 512 tokens)
- **4 layers** (at positions 5, 11, 17, 23) use full global attention

This means most layers only attend to nearby tokens (efficient), while every 6th layer attends to all tokens (captures long-range dependencies).

### Mean Pooling — From Tokens to a Single Vector

The model outputs per-token hidden states of shape `(batch_size, sequence_length, 768)`. To get a **single embedding vector per input**, we apply **mean pooling**:

1. Multiply each token embedding by its attention mask (zero out padding tokens).
2. Sum all token embeddings.
3. Divide by the number of non-padding tokens.

Result: one vector of dimension **768** per input text. Other pooling options include `cls` (first token only), `max` (element-wise maximum), and `lasttoken` (last non-padding token).

<a name="installation"></a>

## 3. Installation

The cell below installs all required packages. It auto-detects your environment:

- **Google Colab**: Installs version-matched `xformers`, `sentencepiece`, `protobuf`, and the core stack (`unsloth`, `peft`, `trl`, `triton`, etc.) with careful dependency pinning to avoid Colab-specific conflicts.
- **Local machine**: Runs `pip install unsloth`, which pulls all dependencies automatically. Make sure you have CUDA-enabled PyTorch installed first (see [Foundations](#foundations) above for install commands).

Two packages are then pinned explicitly:
- `transformers==4.56.2` — ensures compatibility with Unsloth's patches.
- `trl==0.22.2` (installed with `--no-deps`) — the training library, pinned to avoid dependency conflicts.

The `%%capture` magic suppresses the noisy pip output.

In [28]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

<a name="model-loading"></a>

## 4. Model Loading

We load the model using Unsloth's `FastSentenceTransformer`, which wraps Hugging Face's model loading with several optimizations.

**What `from_pretrained` does internally:**

1. **Downloads the model** from Hugging Face Hub (cached at `~/.cache/huggingface/`).
2. **Detects your GPU** and selects the optimal dtype — `bfloat16` on Ampere+ GPUs (compute capability >= 8), `float16` on older hardware like the T4.
3. **Identifies the model as a "fast encoder"** (Gemma-based) and enables the `torch.compile` optimization path.
4. **Wraps the model** in a SentenceTransformer pipeline: Transformer → Pooling → (optional) Normalize.

**Key parameters:**
- **`model_name`** — Any Hugging Face model ID (e.g., `"unsloth/embeddinggemma-300m"`) or a local path.
- **`max_seq_length = 1024`** — Maximum tokens per input. Longer inputs are truncated. The model supports up to 2,048, but 1,024 balances memory and coverage. If you have limited VRAM (< 16 GB), consider reducing to 512.
- **`full_finetuning = False`** — `False` = LoRA (~4% trainable params, much less memory). `True` = all parameters trainable (~10-15x more memory). We use LoRA here.

In [30]:
from unsloth import FastSentenceTransformer

fourbit_models = [
    "unsloth/all-MiniLM-L6-v2",
    "unsloth/embeddinggemma-300m",
    "unsloth/Qwen3-Embedding-4B",
    "unsloth/Qwen3-Embedding-0.6B",
    "unsloth/all-mpnet-base-v2",
    "unsloth/gte-modernbert-base",
    "unsloth/bge-m3"

] # More models at https://huggingface.co/unsloth

model = FastSentenceTransformer.from_pretrained(
    model_name = "unsloth/embeddinggemma-300m",
    max_seq_length = 1024,   # Choose any for long context!
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

==((====))==  Unsloth 2026.3.11: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


AttributeError: module 'huggingface_hub.constants' has no attribute 'HF_HUB_ENABLE_HF_TRANSFER'

<a name="lora"></a>

## 5. LoRA — Theory, Math & Parameter Reference

### The Problem

EmbeddingGemma 300M has ~316 million parameters. Training all of them requires storing parameters in float32, computing gradients for every one, and maintaining optimizer states (Adam stores 2 additional tensors per parameter). This would need roughly 10-15x the model size in GPU memory.

### The LoRA Solution

**LoRA (Low-Rank Adaptation)** freezes the original model weights and injects small trainable matrices alongside them:

```
Original:  output = W @ input
With LoRA: output = W @ input + (B @ A) @ input * scaling
```

Where:
- `A` has shape `(r, in_features)` — the "down-projection"
- `B` has shape `(out_features, r)` — the "up-projection"
- `r` is the **rank** — a small number like 8, 16, 32, or 64
- `scaling = lora_alpha / r`

### The Math, Concretely

We use `r=32` on a projection like `q_proj` which is `(768, 768)`:
- Original `q_proj`: 768 × 768 = 589,824 parameters (frozen)
- LoRA `A`: 32 × 768 = 24,576 parameters (trainable)
- LoRA `B`: 768 × 32 = 24,576 parameters (trainable)
- Total trainable per module: **49,152 parameters** = 8.3% of the original layer

### Why It Works

The key insight from the LoRA paper is that weight updates during fine-tuning are **low-rank** — they live in a much smaller subspace than the full parameter space. By constraining the update to rank `r`, we capture the essential adaptation while dramatically reducing memory and compute.

### Initialization

- `A` is initialized with Kaiming uniform distribution.
- `B` is initialized to **zero**.

This means at the start of training, the LoRA update is zero (`B @ A = 0`) and the model behaves exactly like the pre-trained model. Training gradually moves the update away from zero.

### Target Modules

We target 7 modules in each transformer layer:

| Module | Role | Why Target It |
|--------|------|---------------|
| `q_proj` | Query projection in attention | Controls what the model "asks" about each token |
| `k_proj` | Key projection in attention | Controls what the model "advertises" about each token |
| `v_proj` | Value projection in attention | Controls what information is retrieved during attention |
| `o_proj` | Output projection in attention | Controls how attention outputs are mixed |
| `gate_proj` | Gating in MLP (SwiGLU) | Controls which features are activated |
| `up_proj` | Up-projection in MLP | Expands features for processing |
| `down_proj` | Down-projection in MLP | Compresses features back |

Targeting all 7 adapts both the attention mechanism (how tokens relate) and the feed-forward network (how token representations transform), yielding **4.14% trainable parameters** (13,074,432 / 315,937,536).

### The Scaling Factor

`scaling = lora_alpha / r = 64 / 32 = 2.0`

The LoRA update is multiplied by 2.0 before being added to the original output. A common heuristic is `lora_alpha = 2 * r`. Higher scaling = stronger adaptation (risk of instability); lower = gentler (may need more steps).

### All LoRA Parameters Explained

| Parameter | Value | What It Does |
|-----------|-------|-------------|
| **`r = 32`** | Rank | Bottleneck dimension. 8 = quick experiments, 16 = good default, 32 = complex domains, 64 = max before full fine-tuning. |
| **`lora_alpha = 64`** | Scaling numerator | `scaling = alpha/r`. Common: `alpha = 2*r`. Higher = stronger signal (may need lower LR). |
| **`lora_dropout = 0`** | Dropout rate | Applied before LoRA A matrix. 0.05–0.1 can help on small datasets (<1K). Unsloth optimizes the `0` case. |
| **`bias = "none"`** | Bias training | `"none"` = fastest. `"all"` or `"lora_only"` train bias terms. Minimal impact on quality. |
| **`use_gradient_checkpointing = "unsloth"`** | Memory optimization | `False` = fastest. `True` = standard checkpointing. `"unsloth"` = offloads to CPU pinned memory, ~30% less VRAM for seq >= 512. |
| **`random_state = 3407`** | Random seed | For reproducibility. Any integer works. |
| **`use_rslora = False`** | Rank-Stabilized LoRA | Changes scaling to `alpha / sqrt(r)`. More consistent across rank values. Set `True` when sweeping multiple ranks. |
| **`loftq_config = None`** | LoftQ initialization | Decomposes quantization error into LoRA matrices via SVD. Only useful with `load_in_4bit=True`. |
| **`task_type = "FEATURE_EXTRACTION"`** | PEFT task type | Tells PEFT this is an embedding model (not `CAUSAL_LM` for generation). Always use this for embeddings. |

In [ ]:
model = FastSentenceTransformer.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    task_type = "FEATURE_EXTRACTION"
)

<a name="data"></a>

## 6. Data Preparation

### Dataset Formats for Embedding Models

The format of your data determines which loss function you can use:

| Format | Columns | Compatible Losses |
|--------|---------|-------------------|
| **Positive Pairs** (used here) | anchor, positive | `MultipleNegativesRankingLoss`, `CachedMNRL` |
| **Triplets** | anchor, positive, negative | `TripletLoss`, `MNRL` (treats 3rd col as hard negative) |
| **Pairs + Scores** | text1, text2, float score | `CoSENTLoss`, `CosineSimilarityLoss` |
| **Pairs + Binary Labels** | text1, text2, 0/1 label | `ContrastiveLoss`, `OnlineContrastiveLoss` |
| **Text + Class Label** | text, class_id | `BatchHardTripletLoss`, `BatchAllTripletLoss` |

### The MIRIAD Dataset

We use `tomaarsen/miriad-4.4M-split`, a large-scale collection of 4.4 million medical question-answer pairs distilled from peer-reviewed biomedical literature. Each sample is an **(anchor, positive)** pair — exactly what `MultipleNegativesRankingLoss` expects.

We stream a subset of **10,000 training** and **2,000 evaluation** samples to keep the demo fast.

### Where to Find Training Data

| Dataset | Size | Description |
|---------|------|-------------|
| `sentence-transformers/all-nli` | 557K | NLI pairs (anchor, entailment, contradiction) |
| `sentence-transformers/msmarco-distilbert-base-v4` | 500K | Web search query-passage pairs |
| `tomaarsen/miriad-4.4M-split` | 4.4M | Medical question-passage pairs (used here) |
| `BeIR/*` | Varies | Benchmark datasets for information retrieval |
| `mteb/*` | Varies | Massive Text Embedding Benchmark datasets |

**Synthetic data generation:** If you lack labeled data, feed your documents to an LLM and ask it to generate questions that the document answers, then pair each question with its source document.

### Data Quality Principles

1. **Clean pairs** — every query must actually match its paired document.
2. **Diverse queries** — avoid many near-duplicate rephrasings.
3. **Representative distribution** — training data should match production queries.
4. **Hard negatives help** — plausible-but-incorrect negatives train better than random text.
5. **Deduplicate** — remove exact and near-duplicate pairs before training.

### How Much Data Do You Need?

| Data Size | Expected Outcome |
|-----------|-----------------|
| 100–1,000 pairs | Minimal domain adaptation. May help with specialized terminology. |
| 1,000–10,000 pairs | Good domain adaptation for focused domains. |
| 10,000–100,000 pairs | Strong adaptation. Diminishing returns past this for most tasks. |
| 100,000+ pairs | Useful for very broad domains or significant distribution shifts. |

This notebook uses 10,000 samples and achieves a jump from 5.95% to 88.3% accuracy@1 — demonstrating that moderate amounts of high-quality domain data produce dramatic improvements.

Each sample has two columns: `question` (the medical query) and `passage_text` (the relevant passage). The dataset was loaded in **streaming mode** (`streaming=True`) to avoid downloading all 4.4M rows, then converted to in-memory `Dataset` objects required by the trainer. Let's inspect a sample:

In [ ]:
train_dataset[0]

<a name="baseline"></a>

## 7. Baseline Evaluation

Before fine-tuning, we evaluate the pre-trained model to establish a baseline. The `InformationRetrievalEvaluator` works as follows:

1. Creates a dictionary of **2,000 queries** (the eval questions).
2. Creates a **corpus of 4,000 documents** (2,000 eval passages + 2,000 train passages as distractors).
3. Maps each query to its correct passage via `relevant_docs = {0: [0], 1: [1], ...}`.
4. Encodes all queries and corpus documents, computes cosine similarities, ranks results, and calculates retrieval metrics.

`torch.autocast` enables automatic mixed precision for speed without sacrificing accuracy. The baseline `cosine_accuracy@1` of **~5.95%** tells us the general-purpose model is essentially guessing on this medical domain — exactly the gap fine-tuning will close.

In [ ]:
import torch
from sentence_transformers.evaluation import InformationRetrievalEvaluator

queries = dict(enumerate(eval_dataset["question"]))
corpus = dict(enumerate(list(eval_dataset["passage_text"]) + train_dataset["passage_text"][:2000]))
relevant_docs = {idx: [idx] for idx in queries}
evaluator = InformationRetrievalEvaluator(
    queries = queries,
    corpus = corpus,
    relevant_docs = relevant_docs,
    show_progress_bar = False,
    batch_size = 64,
)
with torch.autocast(device_type = "cuda", dtype = model.dtype, enabled = model.dtype != torch.float16):
    print(evaluator(model))

<a name="loss-functions"></a>

## 8. Loss Functions — The Complete Catalog

Before configuring training, let's understand the loss function we'll use and the alternatives available.

### Choosing the Right Loss — A Decision Tree

```
Do you have (anchor, positive) pairs?
  YES → MultipleNegativesRankingLoss (start here)
        - Need bigger effective batch? → CachedMultipleNegativesRankingLoss
        - Symmetric task? → MultipleNegativesSymmetricRankingLoss
        - Have false-negative problems? → GISTEmbedLoss

Do you have similarity scores?
  YES → CoSENTLoss (recommended over CosineSimilarityLoss)

Do you have binary similar/dissimilar labels?
  YES → OnlineContrastiveLoss

Do you have explicit (anchor, positive, negative) triplets?
  YES → MultipleNegativesRankingLoss (still best default) or TripletLoss

Do you have class labels?
  YES → BatchHardTripletLoss

Do you have a teacher model?
  YES → DistillKLDivLoss or MSELoss
```

### MultipleNegativesRankingLoss (MNRL) — Used in This Notebook

This is the most popular loss for embedding fine-tuning. Here's how it works step by step:

1. Take a batch of N `(anchor, positive)` pairs.
2. Encode all anchors → N anchor embeddings.
3. Encode all positives → N positive embeddings.
4. Compute an **N × N cosine similarity matrix** between all anchors and all positives.
5. The correct positive for anchor `i` is at index `i` (the diagonal).
6. Apply **cross-entropy loss**: for each anchor, the model should assign the highest probability to its correct positive.

**Why it's effective:** Every positive in the batch serves as a negative for all other anchors. With `batch_size=64`, each anchor gets 1 positive and 63 negatives — no manual negative mining needed.

**Key parameter:** `scale = 20.0` — multiplied by similarity scores before softmax (temperature = 1/20 = 0.05). Higher scale = sharper probability distribution = harder training signal.

### Other Notable Losses

| Loss | When to Use |
|------|-------------|
| **CachedMNRL** | Same as MNRL but uses GradCache to support very large effective batches (512+) with same GPU memory. 2-2.4x slower. |
| **CoSENTLoss** | When you have continuous similarity scores. Preserves score ordering. |
| **GISTEmbedLoss** | When your corpus has many semantically similar documents (uses a guide model to filter false negatives). |

### Meta-Losses (Wrappers)

- **`MatryoshkaLoss`** — Trains at multiple embedding dimensions (768, 512, 256, 128, 64) simultaneously. After training, you can truncate embeddings to smaller dimensions with graceful degradation — useful for trading storage vs. quality.
- **`AdaptiveLayerLoss`** — Trains intermediate layers to produce good embeddings. After training, you can use fewer transformer layers for faster inference.

<a name="train"></a>

## 9. Training Configuration

Now we configure and launch training. We use `MultipleNegativesRankingLoss` as our loss function (see decision tree above) and set up the `SentenceTransformerTrainer` with carefully chosen arguments.

**Every training argument explained:**

| Argument | Value | What It Does |
|----------|-------|-------------|
| `max_steps = 30` | Total optimizer steps. For a quick demo; use `num_train_epochs=1` and `max_steps=None` for a full run. With our data: effective batch 128, so 30 steps ≈ 38% of one epoch. |
| `per_device_train_batch_size = 64` | Samples per GPU per forward pass. For MNRL, **batch size directly affects quality** because other samples serve as negatives. Use the largest that fits in memory. |
| `gradient_accumulation_steps = 2` | Accumulates gradients over 2 forward passes before one optimizer step. Effective batch = 64 × 2 = 128. **Important nuance:** GA does NOT increase in-batch negatives — each forward pass still has only 64 negatives. Use `CachedMNRL` if you need more. |
| `learning_rate = 2e-5` | Max LR for AdamW. Typical LoRA range: 1e-5 to 5e-5. If loss oscillates, reduce; if loss barely moves, increase. |
| `warmup_ratio = 0.03` | Fraction of steps spent linearly ramping LR from 0. LoRA matrices start at zero, so gentle early updates prevent instability. |
| `lr_scheduler_type = "linear"` | Linear decay after warmup. `"cosine"` is often slightly better for longer runs. |
| `prompts = {...}` | Maps dataset columns to model prompt prefixes. EmbeddingGemma uses different prefixes for queries vs. documents (**asymmetric embedding**), so the same text gets different embeddings depending on its role. |
| `batch_sampler = NO_DUPLICATES` | Ensures no two samples in a batch share column values. Prevents MNRL from treating identical texts as negatives. |
| `eval_strategy = "steps"`, `eval_steps = 5` | Run evaluation every 5 steps (6 checkpoints over 30 steps). Helps monitor for overfitting. |
| `bf16 = is_bf16_supported()` | Uses bfloat16 mixed precision if available. bfloat16 has the same range as float32 (more stable than float16). |
| `report_to = "none"` | No external logging. Use `"wandb"` or `"tensorboard"` for experiment tracking in production. |

In [ ]:
from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses
)
from sentence_transformers.training_args import BatchSamplers
from unsloth import is_bf16_supported

# This will use other positives in the same batch as negative examples
loss = losses.MultipleNegativesRankingLoss(model)

trainer = SentenceTransformerTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    loss = loss,
    args = SentenceTransformerTrainingArguments(
        # num_train_epochs = 1,
        max_steps = 30,
        per_device_train_batch_size = 64,
        per_device_eval_batch_size = 64,
        gradient_accumulation_steps = 2, # Use GA to mimic batch size!
        learning_rate = 2e-5,
        logging_steps = 5,
        warmup_ratio = 0.03,
        prompts = {  # Map training column names to model prompts
          "question": model.prompts["query"],
          "passage_text": model.prompts["document"],
        },
        report_to = "none", # Use TrackIO/WandB etc
        eval_strategy = "steps",
        eval_steps = 5,
        bf16 = is_bf16_supported(),
        output_dir = "output",
        lr_scheduler_type = "linear",
        # Because we have duplicate anchors in the dataset, we don't want
        # to accidentally use them for negative examples
        batch_sampler = BatchSamplers.NO_DUPLICATES,
    ),
    evaluator = evaluator, # Optional, will make training slower
)

In [ ]:
# @title Show current memory stats
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

<a name="training-internals"></a>

## 10. Training Internals — What Happens Under the Hood

Before we launch training, here's what happens at each step:

**Step 1 — Data Loading:** The `NoDuplicatesBatchSampler` selects 64 non-overlapping samples. The data collator prepends prompt prefixes (query prompt for questions, document prompt for passages), then tokenizes.

**Step 2 — Forward Pass:** Tokens pass through the embedding layer (262,144 vocab → 768 dim), then through all 24 transformer layers with LoRA adapters active. Mean pooling reduces `(batch, seq_len, 768)` → `(batch, 768)`.

**Step 3 — Loss (MNRL):** Compute 64×64 cosine similarity matrix, multiply by `scale=20.0`, apply softmax row-wise, compute cross-entropy against the diagonal (correct positives).

**Step 4 — Backward Pass:** Gradients flow back through the loss, pooling, and all 24 layers. Gradient checkpointing recomputes activations during backprop. Only LoRA A and B matrices receive updates.

**Step 5 — Gradient Accumulation:** Repeat steps 1–4 for a second mini-batch; sum gradients.

**Step 6 — Optimizer Step:** AdamW updates LoRA parameters using accumulated gradients. The LR scheduler adjusts the learning rate.

**Step 7 — Logging/Eval:** Every 5 steps, the evaluator encodes all queries and corpus, ranks results, and reports metrics.

### What Unsloth Optimizes

1. **LoRA Forward Pass** — Fused CUDA kernels combine `dropout → A → B → scale` operations for SwiGLU patterns.
2. **Gradient Checkpointing** — Activations are asynchronously copied to CPU pinned memory via CUDA streams, then copied back during the backward pass (the `"unsloth"` mode).
3. **`torch.compile`** — Compiles the forward pass into optimized Triton kernels, reducing Python overhead and enabling kernel fusion. Works automatically on standard NVIDIA GPUs (T4, RTX 3000/4000/5000, A100, H100).

### Memory Breakdown

| GPU | VRAM | Model + LoRA | Training Peak | Notes |
|-----|------|-------------|---------------|-------|
| T4 (Colab free) | 15 GB | ~7.7 GB | ~11.3 GB | Tight — batch 64 works, 128 may OOM |
| L4 / RTX 4090 | 24 GB | ~3.4 GB | ~7 GB | Comfortable — batch 128 fits |
| RTX 3060 | 12 GB | ~3.4 GB | ~8 GB | Tight — batch 32, use gradient checkpointing |
| A100 40 GB | 40 GB | ~3.4 GB | ~7 GB | Abundant — batch 256+ or longer sequences |
| A100 80 GB / H100 | 80 GB | ~3.4 GB | ~7 GB | Massive — batch 512+ feasible |

If you hit OOM, reduce `per_device_train_batch_size` first, then `max_seq_length`. Let's train:

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="post-eval"></a>

## 11. Post-Training Evaluation

Now let's run the same evaluator on our fine-tuned model and compare against the baseline:

In [ ]:
with torch.autocast(device_type = "cuda", dtype = model.dtype, enabled = model.dtype != torch.float16):
    print(evaluator(model))

<a name="metrics-explained"></a>

## 12. Evaluation Metrics Explained

Let's unpack what each metric means:

| Metric | Question It Answers | Interpretation |
|--------|-------------------|----------------|
| **Accuracy@k** | "Is the correct document in the top k?" | Most intuitive. If your search shows 10 results, accuracy@10 = how often the user sees the right answer. |
| **Precision@k** | "What fraction of top-k results are relevant?" | With 1 relevant doc per query, precision@3 ≈ 1/3 when correct doc is in top 3. |
| **Recall@k** | "What fraction of relevant docs are in top k?" | With 1 relevant doc per query, equals accuracy@k. |
| **MRR@k** (Mean Reciprocal Rank) | "How high is the correct doc ranked, on average?" | Reciprocal rank: #1 → 1.0, #2 → 0.5, #3 → 0.333. MRR = average across all queries. |
| **NDCG@k** (Normalized Discounted Cumulative Gain) | "How well-ordered are the results?" | Penalizes relevant docs appearing lower. 1.0 = perfect ranking. |
| **MAP@k** (Mean Average Precision) | "How precise is the full ranking?" | For each relevant doc found, compute precision at that rank, then average. |

### Before vs. After Fine-Tuning

| Metric | Before Training | After Training | Improvement |
|--------|----------------|----------------|-------------|
| accuracy@1 | 0.0595 | 0.883 | +1,384% |
| accuracy@10 | 0.749 | 0.9795 | +31% |
| NDCG@10 | 0.433 | 0.935 | +116% |
| MRR@10 | 0.329 | 0.920 | +180% |

The pre-trained model performed essentially randomly on this medical domain (5.95% accuracy@1). After just 30 steps of fine-tuning, it became highly accurate (88.3%). This demonstrates the power of domain-specific fine-tuning.

### Which Metric to Optimize?

- **Primary:** NDCG@10 (captures ranking quality).
- **Secondary:** Accuracy@1 or MRR@10 (top-result quality).
- **Sanity check:** Accuracy@10 should be very high — if it's low, the model is fundamentally broken.

<a name="inference"></a>

## 13. Inference

Let's use the trained model for real-world retrieval. The workflow is:

1. **Encode the query** — `model.encode()` tokenizes, runs the forward pass, and applies mean pooling to produce a 768-dim vector.
2. **Encode the candidates** — same process, but with the document prompt prefix.
3. **Compute similarity** — `model.similarity()` calculates cosine similarity between the query vector and each candidate vector.
4. **Rank** — sort by descending similarity.

**Production tips:**
- **Batch your queries:** Encoding one at a time is wasteful. Batch 32–128 queries for throughput.
- **Pre-compute corpus embeddings:** Encode your entire corpus offline and store in a vector database (FAISS, Pinecone, Weaviate, etc.).
- **GPU inference:** Keep the model on GPU. CPU inference is 10-50x slower.
- **ONNX / TensorRT:** Export for faster inference outside Python.

In [ ]:
query = "Patient presents with sharp chest pain that improves when leaning forward."

candidates = [
    "Acute Pericarditis often involves pleuritic chest pain relieved by sitting up and leaning forward.",
    "Myocardial Infarction typically presents with crushing substernal pressure and radiation to the left arm.",
    "Pneumothorax is characterized by sudden onset shortness of breath and unilateral chest pain.",
    "Gastroesophageal Reflux Disease (GERD) causes burning retrosternal pain usually after meals."
]

query_emb = model.encode(query, convert_to_tensor = True)
candidate_embs = model.encode(candidates, convert_to_tensor = True)
similarities = model.similarity(query_emb, candidate_embs)

ranking = similarities.argsort(descending = True)[0]

for idx in ranking.tolist():
    score = similarities[0][idx].item()
    text = candidates[idx]
    print(f"{score:.4f} | {text}")

<a name="saving-overview"></a>

## 14. Saving & Deployment Overview

We have several options for exporting our fine-tuned model, depending on the deployment target:

| Deployment Target | Recommended Format |
|-------------------|--------------------|
| Python inference (same machine) | LoRA adapters or merged 16-bit |
| Python inference (production server) | Merged 16-bit |
| llama.cpp / Ollama / LM Studio | GGUF `q4_k_m` or `q5_k_m` |
| Sharing on Hugging Face | Merged 16-bit + GGUF |
| Continued fine-tuning | LoRA adapters |
| Mobile / edge devices | GGUF `q4_k_m` or `q3_k_m` |

### What Is Merging?

When we "merge" LoRA weights, we mathematically fold the adapter back into the base model:

```
W_merged = W_base + (B @ A) * scaling
```

The result is a single standard model with no adapter overhead at inference time. The adapter cannot be separated again after merging.

<a name="save-lora"></a>

### Option 1: LoRA Adapters Only (Smallest, Most Flexible)

Saves only the LoRA adapter weights (~50-100 MB) plus the tokenizer. The base model is NOT included — you'll need to load it separately and apply the adapters at inference time.

**Pros:** Smallest files. Can swap adapters at runtime. Easy to version control. Can continue fine-tuning later.
**Cons:** Requires loading base model + adapter separately.

Use `push_to_hub` for Hugging Face upload, or `save_pretrained` for local saving. To save to 16-bit, scroll down.

**[NOTE]** This ONLY saves the LoRA adapters, not the full model.

In [ ]:
model.save_pretrained("embeddinggemma_lora")  # Local saving
model.tokenizer.save_pretrained("embeddinggemma_lora")
# model.push_to_hub("your_name/embeddinggemma_lora", token = "YOUR_HF_TOKEN") # Online saving
# model.tokenizer.push_to_hub("your_name/embeddinggemma_lora", token = "YOUR_HF_TOKEN") # Online saving

### Loading Saved LoRA Adapters

To load the LoRA adapters we just saved for inference, change `False` to `True` below. Unsloth's `from_pretrained` automatically detects the adapter format and applies it on top of the base model:

In [ ]:
if False:
    from unsloth import FastSentenceTransformer
    model = FastSentenceTransformer.from_pretrained(
        "model",
    )

<a name="save-merged"></a>

### Option 2: Merged Float16 (Full Model, No Adapter Separation)

This merges LoRA weights into the base model (`W_merged = W_base + B@A * scaling`) and saves as a standard float16 model (~600 MB).

**Pros:** Standard model format. No LoRA loading overhead. Required for GGUF conversion.
**Cons:** Larger files. Cannot separate the adapter later.

Select `merged_16bit` for float16, `merged_4bit` for int4 (causes accuracy loss — use only for memory-constrained inference). Use `push_to_hub_merged` to upload to Hugging Face. See the [deployment docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more options.

In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("embeddinggemma_finetune_16bit", tokenizer = model.tokenizer, save_method = "merged_16bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/embeddinggemma_finetune_16bit", tokenizer = model.tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("embeddinggemma_lora")
if False: # Pushing to HF Hub
    model.push_to_hub("HF_USERNAME/embeddinggemma_lora", token = "YOUR_HF_TOKEN")

<a name="save-gguf"></a>

### Option 3: GGUF / llama.cpp Conversion

GGUF is the file format used by llama.cpp, Ollama, and LM Studio. We clone `llama.cpp` and quantize the model. Default quantization is `q8_0`.

**Available quantization methods:**

| Method | Quality | Size | Recommendation |
|--------|---------|------|----------------|
| `f16` | Lossless | Large | Reference only |
| `q8_0` | Near-lossless | Medium | Good default |
| `q6_k` | Excellent | Smaller | Quality-focused |
| **`q5_k_m`** | Very good | Small | **Recommended balance** |
| **`q4_k_m`** | Good | Smallest practical | **Recommended for production** |
| `q4_k_s` | Good | Smallest | Slightly less quality than q4_k_m |
| `q3_k_m` | Acceptable | Tiny | Quality-constrained environments |
| `q2_k` | Degraded | Tiniest | Not recommended for embeddings |

You can save multiple quantizations at once for efficiency (see the second `push_to_hub_gguf` call below).

In [ ]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf("embeddinggemma_finetune",)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False:
    model.push_to_hub_gguf("HF_USERNAME/embeddinggemma_finetune", token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("embeddinggemma_finetune", quantization_method = "f16")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/embeddinggemma_finetune", quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("embeddinggemma_finetune", quantization_method = "q4_k_m")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/embeddinggemma_finetune", quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/embeddinggemma_finetune", # Change hf to your username!
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

<a name="production"></a>

## 15. Production Considerations

### Evaluation Beyond Training Metrics

The notebook's evaluator uses a synthetic setup. For production, you also need:
- **Held-out test set** — data the model never saw during training or hyperparameter selection.
- **Domain-specific benchmarks** — e.g., medical QA benchmarks for a medical retrieval system.
- **Human evaluation** — domain experts rating retrieval quality on real queries.
- **A/B testing** — compare your fine-tuned model against the baseline with real users.

### Data Contamination

Ensure your test set has no overlap with training data. Even partial overlap (same document, different question) can inflate metrics.

### Embedding Storage at Scale

768-dimensional float32 embeddings = 3,072 bytes per document. At scale: 1M docs ≈ 2.9 GB, 100M docs ≈ 286 GB.

**Strategies:** Use `MatryoshkaLoss` then truncate to 256 dims. Quantize embeddings to int8 (4x reduction). Use approximate nearest neighbor search (FAISS, ScaNN).

### Inference Optimization

- **Batch queries** — encode 32–128 at a time, not one by one.
- **Pre-compute corpus embeddings** — encode offline, store in a vector database.
- **ONNX / TensorRT** — export for faster inference outside Python.
- **GPU inference** — 10-50x faster than CPU.

### When to Retrain

- Domain shifts (new terminology, new document types).
- Significantly more training data collected (2x+).
- Eval metrics on fresh data degrade.
- Retrieval task changes (e.g., Q&A → document-to-document similarity).

### Hyperparameter Search Checklist

For production models, sweep these parameters:

| Parameter | Values to Try | Priority |
|-----------|--------------|----------|
| `r` (LoRA rank) | 8, 16, 32, 64 | High |
| `learning_rate` | 1e-5, 2e-5, 5e-5 | High |
| `per_device_train_batch_size` | 32, 64, 128 | High |
| `num_train_epochs` | 1, 3, 5 | High |
| `max_seq_length` | 256, 512, 1024 | Medium |
| `lora_alpha` | r, 2*r | Medium |
| Loss function | MNRL, CachedMNRL, CoSENT | Medium |
| `warmup_ratio` | 0.01, 0.03, 0.1 | Low |
| `lr_scheduler_type` | linear, cosine | Low |
| `lora_dropout` | 0, 0.05, 0.1 | Low |

### Full Fine-Tuning vs. LoRA

Consider full fine-tuning (`full_finetuning=True`) when: >50K high-quality pairs, LoRA at rank 64 still underperforms, ample GPU memory (10x model size), target domain very different from pre-training data.

LoRA is preferred when: limited data (<50K pairs), fast iteration needed, multiple domain adapters on one base model, memory constrained.

<a name="troubleshooting"></a>

## 16. Troubleshooting Common Issues

| Issue | Cause | Fix |
|-------|-------|-----|
| **`NotImplementedError: Unsloth cannot find any torch accelerator`** | CPU-only PyTorch build installed. | Install CUDA-enabled PyTorch: `pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124` (replace `cu124` with your CUDA version — see `nvidia-smi`). Verify with `torch.cuda.is_available()`. |
| **`RuntimeError: CUDA error: no kernel image is available`** | PyTorch compiled for different GPU architecture than yours. | Install latest PyTorch: `pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu124`. |
| **`OutOfMemoryError` (GPU memory)** | Batch too large, sequence too long, or model too big for your GPU. | Reduce `per_device_train_batch_size` (64→32→16). Increase `gradient_accumulation_steps`. Reduce `max_seq_length`. Use `"unsloth"` gradient checkpointing. Try `load_in_4bit=True`. |
| **Training loss is NaN or exploding** | Learning rate too high, bad data, or float16 overflow. | Reduce `learning_rate` (try 1e-5). Increase `warmup_ratio` (try 0.1). Check data for empty strings. Use `bf16=True` if GPU supports it (Ampere+). |
| **Training loss decreases but eval metrics get worse (overfitting)** | Too many steps, too little data, or rank too high. | Reduce `max_steps`/epochs. Increase `lora_dropout` (0.05–0.1). Reduce rank `r`. Add more data. Use early stopping. |
| **`ModuleNotFoundError: No module named 'sentence_transformers'`** | Missing dependency. | `pip install sentence-transformers` |
| **Model produces identical embeddings for different inputs** | Training collapsed, or model in wrong mode. | Call `model.eval()` for inference. If training collapsed, reduce learning rate and check data quality. |
| **Colab session disconnects during training** | Idle timeout or session time limit. | Set `save_steps=10` in training args. Resume with `trainer.train(resume_from_checkpoint=True)`. Consider Colab Pro for longer sessions. |

<a name="glossary"></a>

## 17. Glossary

| Term | Definition |
|------|-----------|
| **Adapter** | Small trainable module inserted into a frozen pre-trained model (e.g., LoRA). |
| **Attention** | Mechanism that lets each token "look at" other tokens to gather context. |
| **bfloat16** | 16-bit floating point format with same exponent range as float32, used for stable training. Requires compute capability >= 8.0 (Ampere+). |
| **Bidirectional** | Attention that looks at all tokens (past and future), not just past tokens. |
| **Causal** | Attention that only looks at past tokens, used in generative models. |
| **Compute Capability** | NVIDIA's version number for GPU features; determines bfloat16 support (>= 8.0), Flash Attention availability, etc. |
| **Contrastive Learning** | Training approach that brings similar items closer and pushes dissimilar items apart in embedding space. |
| **Cosine Similarity** | Measure of angle between two vectors; 1.0 = identical direction, 0 = perpendicular, -1 = opposite. |
| **CUDA** | NVIDIA's parallel computing platform. PyTorch must be installed with a matching CUDA version. |
| **Embedding** | Fixed-length vector representation of text. |
| **Epoch** | One complete pass through all training data. |
| **GGUF** | File format for quantized models used by llama.cpp and Ollama. |
| **Gradient Accumulation** | Summing gradients over multiple mini-batches before updating weights. |
| **Gradient Checkpointing** | Trading compute for memory by recomputing activations during backpropagation. |
| **GQA** | Grouped Query Attention — variant with fewer key-value heads than query heads, reducing memory. |
| **In-batch Negatives** | Using other samples in the same batch as negative examples for contrastive learning. |
| **LoRA** | Low-Rank Adaptation — fine-tuning method that trains small rank-decomposed matrices alongside frozen weights. |
| **Mean Pooling** | Averaging all token embeddings (excluding padding) to get one vector per input. |
| **MLP** | Multi-Layer Perceptron — the feed-forward network in each transformer layer. |
| **MNRL** | MultipleNegativesRankingLoss — contrastive loss using in-batch negatives. |
| **NDCG** | Normalized Discounted Cumulative Gain — ranking quality metric. |
| **PEFT** | Parameter-Efficient Fine-Tuning — umbrella term for methods like LoRA. |
| **Quantization** | Reducing numerical precision of model weights (e.g., float16 to int4) to save memory. |
| **Rank (r)** | Dimensionality of the LoRA bottleneck; controls capacity vs. efficiency tradeoff. |
| **RMSNorm** | Root Mean Square Normalization — layer normalization variant used in modern transformers. |
| **RoPE** | Rotary Position Embeddings — method to encode token positions in attention. |
| **Scaling** | Multiplier applied to LoRA updates: `lora_alpha / r`. |
| **SDPA** | Scaled Dot-Product Attention — PyTorch's optimized attention implementation. |
| **Sliding Window Attention** | Attention limited to a fixed window of nearby tokens for efficiency. |
| **SwiGLU** | Gated activation function: `GELU(gate(x)) * value(x)`, used in modern MLPs. |
| **torch.compile** | PyTorch's JIT compiler that converts Python code to optimized Triton/CUDA kernels. |
| **VRAM** | Video RAM — the GPU's dedicated memory; determines max batch size and model size. |
| **Warmup** | Gradually increasing learning rate from 0 during early training steps. |

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>